# Argus — plate detector training (Kaggle GPU)

Trains the single-class Indian license-plate detector on the
[Indian number plate dataset by Quobotic](https://universe.roboflow.com/quobotic/indian-number-plate)
— 2,573 images, one class, CC BY 4.0.

Set `EXTRA_DATASETS` in Step 1 to merge further sources into the same run;
see `ml/TRAINING.md`, "Route C".

This is the **primary** training path. The local CPU fallback (`--cpu`) is in
`ml/TRAINING.md`, Route B, and at this dataset size takes about an hour.

## Before you run anything

| Setting (right panel) | Value |
|---|---|
| **Accelerator** | GPU T4 x2 (or P100) |
| **Internet** | **On** — required to pip install, clone the repo, and reach Roboflow |
| **Add-ons → Secrets** | a secret named `ROBOFLOW_API_KEY` |

A new or unverified Kaggle account cannot enable GPU or Internet — both toggles
stay greyed out. Settings → Phone verification fixes it, once.

Get the Roboflow key from <https://app.roboflow.com> → Settings → API Keys →
Private API Key. Store it as a Kaggle secret rather than pasting it into a
cell: notebooks get shared, and a key pasted into a cell gets shared with it.

## Step 1 — Configure

The dataset is downloaded by the cell below; nothing needs attaching through
"Add Data".

In [ ]:
WORKSPACE = "quobotic"
PROJECT   = "indian-number-plate"
VERSION   = 3
DEST      = "/kaggle/working/plates"
EPOCHS    = 60

# MERGE MORE DATA (optional). Paths to further dataset roots already present in
# the session -- attached through "Add Data", or downloaded in a cell of your
# own. Roboflow's own export is always included; these are added to it.
#
# ml/prepare_dataset.py reads YOLO, COCO or Pascal VOC and writes one merged
# YOLO set, dropping photographs that appear in more than one source. That
# deduplication is the point: these datasets are largely re-uploads of each
# other, and the same image in train and val inflates val mAP silently.
#
# ml/TRAINING.md lists candidates with sizes and licences.
EXTRA_DATASETS = []          # e.g. ["/kaggle/input/indian-number-plate-images"]

# CONTINUE FROM THE SHIPPED DETECTOR instead of stock COCO weights.
#
# Only useful with EXTRA_DATASETS: fine-tuning on the same data the model was
# trained on teaches it nothing. Needs runs/detect/plate/weights/best.pt in the
# session -- attach it as a Kaggle Dataset, it is not in the repo.
#
# LR0 matters more than it looks. Ultralytics defaults to 0.01, right for a
# random head and large enough to walk a trained detector away from what it
# learned. Around 0.002 keeps it.
FINETUNE_FROM = None         # e.g. "/kaggle/input/argus-weights/best.pt"
LR0           = 0.002

# 2,573 images from Roboflow alone. patience=15 in train_plate.py will often
# stop the run before 60 epochs at that size -- that is the mechanism working,
# not a failure. Merged sets are larger and usually run longer.


## Step 2 — Confirm the GPU and install

`lap` is BoT-SORT's assignment solver. Installing it here rather than letting
Ultralytics AutoUpdate it mid-run avoids a pip install firing in the middle of
training.

In [ ]:
!nvidia-smi
!pip -q install ultralytics lap

## Step 3 — Clone the repo

The notebook calls `ml/prepare_dataset.py` and `ml/train_plate.py` from the
repo. **No training code is duplicated here**, so the Kaggle path and the local
path cannot drift apart.

In [ ]:
%cd /kaggle/working
!rm -rf Argus && git clone -q https://github.com/Deeptanshu789/Argus.git
%cd /kaggle/working/Argus

## Step 4 — Get the dataset

Roboflow exports in **YOLOv8 format**: one `.txt` label file beside each image,
in per-split directories, plus a `data.yaml`. That is already what Ultralytics
trains on, so `prepare_dataset.py` is not needed on this path — it exists for
COCO and Pascal VOC sets.

It also removes the failure that produced the first Argus model. The previous
dataset shipped one `_annotations.coco.json` per split, all at the archive
root; unzipped into a single folder they overwrote each other, and the model
trained on 1,365 of 8,823 images while the warning scrolled past in the log.
Per-image label files cannot collide that way.

The checks below still count images against labels anyway. "Cannot fail this
way" is not "cannot fail".

In [ ]:
!pip -q install roboflow

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
ds = (Roboflow(api_key=key)
      .workspace(WORKSPACE).project(PROJECT).version(VERSION)
      .download("yolov8", location=DEST))
ROOT = ds.location
DATA = f"{ROOT}/data.yaml"
print(ROOT)

## Step 5 — Check the download, then fix the paths

Two separate problems, both silent.

**Counts.** An image with no label file is not skipped — YOLO trains on it as
"there is no plate here", which teaches the model to miss.

**Paths.** Roboflow writes `train: ../train/images`, relative to the working
directory rather than to the file. Run from anywhere else, Ultralytics looks in
the wrong place, warns once, and trains on an empty split.

In [ ]:
import glob, os, yaml

total = 0
for split in ("train", "valid", "test"):
    imgs = glob.glob(f"{ROOT}/{split}/images/*")
    labs = glob.glob(f"{ROOT}/{split}/labels/*.txt")
    print(f"{split:6} {len(imgs):5} images  {len(labs):5} labels")
    assert len(imgs) == len(labs), f"{split}: {len(imgs)} images, {len(labs)} labels"
    total += len(imgs)

print("total", total)
# Version 3 is 2,573 images. May exceed that if the version applies
# augmentation; below it means a split failed to download.
assert total >= 2573, f"expected at least 2573 images, found {total}"

cfg = yaml.safe_load(open(DATA))
assert cfg["nc"] == 1, f"expected one class, got {cfg['nc']}: {cfg['names']}"

cfg["path"] = ROOT
for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    if split in cfg:
        cfg[split] = f"{ROOT}/{folder}/images"
yaml.safe_dump(cfg, open(DATA, "w"))
print(open(DATA).read())

for split in ("train", "val", "test"):
    d = cfg.get(split)
    if d:
        assert os.path.isdir(d) and os.listdir(d), f"{split} -> {d} missing or empty"
print("paths OK")

## Step 6 — Train

GPU defaults from `ml/train_plate.py`: `imgsz=640, batch=32, amp=True,
freeze=0`.

Note `freeze=0` — the backbone is **not** frozen. Freezing it is a CPU
concession that costs accuracy, and on a T4 there is no reason to pay it.
With `EXTRA_DATASETS` set, the cell below merges first and trains on the union.
Read the merge output before the training starts: the duplicate count and the
per-source image counts are where a bad download shows up.

With `FINETUNE_FROM` set it continues from those weights at `LR0` instead of
starting from COCO.


In [ ]:
import subprocess, sys

if EXTRA_DATASETS:
    MERGED = "/kaggle/working/plates-merged"
    # Roboflow's export first, so its annotations win any image the others also
    # carry -- prepare_dataset.py keeps the earliest --src on a duplicate.
    subprocess.run(
        [sys.executable, "ml/prepare_dataset.py",
         "--src", ROOT, *EXTRA_DATASETS,
         "--dst", MERGED, "--subset", "30000", "--val", "2000"],
        check=True)
    TRAIN_DATA = f"{MERGED}/data.yaml"
else:
    TRAIN_DATA = DATA

cmd = [sys.executable, "ml/train_plate.py",
       "--data", TRAIN_DATA, "--epochs", str(EPOCHS), "--device", "0"]
if FINETUNE_FROM:
    cmd += ["--model", FINETUNE_FROM, "--lr0", str(LR0)]

print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)


## Step 7 — Read the score

**Go/no-go bar: `mAP50 >= 0.85`.**

This is a score on *this dataset's* validation split. It is not comparable to
the 0.928 quoted for the shipped weights, which was measured on a different
set. The only comparable number is `ml/score_plates.py`, run on the laptop
against the same 45 hand-labelled plates for both models — see
`ml/TRAINING.md`, "Decide with numbers".

In [ ]:
import csv, pathlib

R = pathlib.Path("runs/detect/plate")
rows = list(csv.DictReader(open(R / "results.csv")))
last = {k.strip(): v for k, v in rows[-1].items()}
m50   = float(last["metrics/mAP50(B)"])
m5095 = float(last["metrics/mAP50-95(B)"])
print(f"epochs run : {len(rows)}")
print(f"mAP50      : {m50:.3f}")
print(f"mAP50-95   : {m5095:.3f}\n")

if m50 >= 0.85:
    print("PASS — export the weights (Step 8).")
elif m50 >= 0.70:
    print("MARGINAL — check the counts in Step 5 before re-running; a GPU run costs minutes.")
else:
    print("FAIL — this is almost certainly a DATA problem, not a training one.")
    print("Single-class, tight-boxed plates reach 0.85 readily; more epochs will")
    print("not fix a bad conversion. Re-check Step 4: labels must be class 0 with")
    print("normalized xywh boxes.")

In [ ]:
from IPython.display import Image, display
display(Image(f"{R}/results.png"))

## Step 8 — Export the weights

Download `argus-plate-weights.zip` from the **Output** panel on the right.

Install it into `runs/detect/plate-new`, **not** over `runs/detect/plate`. A
higher mAP does not mean a better system, in either direction. The full retrain
that took mAP50 from 0.928 to 0.991 scores exactly LEVEL with the shipped
weights on plates actually read correctly -- 39 of 45 for both. mAP moved and
`correct` did not. Score both on the laptop and swap only on `correct`, and
re-sweep `PLATE_PAD` first: a detector that has moved crops differently, and
the crop is what OCR sees. `ml/TRAINING.md` has the procedure.

In [ ]:
!cd runs/detect/plate/weights && zip -q /kaggle/working/argus-plate-weights.zip best.pt last.pt
!ls -lh /kaggle/working/argus-plate-weights.zip

## If the session died mid-run

Kaggle kills long sessions. Re-run Steps 2, 3, 4 and 5, then uncomment and run
this — it picks up from the last checkpoint rather than starting over.

In [ ]:
# !python ml/train_plate.py --data {DATA} --epochs {EPOCHS} --device 0 --resume

## Done — next steps are on the laptop

```bash
cd ~/code/Argus && git pull
mkdir -p runs/detect/plate-new/weights
unzip ~/Downloads/argus-plate-weights.zip -d runs/detect/plate-new/weights
./.venv/bin/python ml/export_onnx.py \
    --weights runs/detect/plate-new/weights/best.pt --fp32
```

`--fp32` because it needs nothing but the weights, and measures 9 ms/frame on
the build machine against a 50 ms budget. int8 additionally needs calibration
images, which live in the dataset, not on the laptop.

Then decide whether it is actually better, re-measuring the crop padding first:

```bash
for p in 0.04 0.08 0.14 0.20; do
  ./.venv/bin/python ml/score_plates.py \
      --model runs/detect/plate-new/weights/best.pt --pad $p | grep -E "correct|wrong"
done
```

Swap only if `correct` beats the shipped 35 of 45.